# 02 — Collecte des publications Facebook

Explique la collecte Apify des pages ENSEA, clubs et associations, puis contrôle les dates, permaliens et doublons.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\.venv\Scripts\python.exe
Données actives : <PROJECT_ROOT>\data


In [2]:
import json
import os
import pandas as pd
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
facebook = config.facebook_scraping
rows = []
for path in sorted(facebook.raw_full_path.glob('*.jsonl')):
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            rows.append(json.loads(line))
posts = pd.DataFrame(rows)
print('Publications normalisées :', len(posts))
if not posts.empty:
    display(posts.groupby('page_key').size().rename('publications').to_frame())
    display(posts[['page_key', 'published_at', 'post_url', 'text']].head())


Publications normalisées : 232


,publications
page_key,
AES_page,28
CDR_page,30
DAS_page,26
DISE_page,30
ECOSTAT_page,30
EDSC_page,29
EJS_page,29
ENSEA_page,30


,page_key,published_at,post_url,text
0,AES_page,2026-03-25T19:33:19+00:00,https://www.facebook.com/permalink.php?story_f...,🎉✨ 𝐅𝐄́𝐋𝐈𝐂𝐈𝐓𝐀𝐓𝐈𝐎𝐍𝐒 𝐚̀ 𝐥’𝐚𝐢̂𝐧𝐞́ 𝐀𝐃𝐉𝐎𝐔𝐌𝐀𝐍𝐈 𝐉𝐨𝐬𝐮𝐞́...
1,AES_page,2026-03-20T18:53:02+00:00,https://www.facebook.com/permalink.php?story_f...,"En ce jour de célébration, 𝐥’𝐀𝐄𝐒, 𝐞𝐧 𝐩𝐚𝐫𝐭𝐞𝐧𝐚𝐫𝐢..."
2,AES_page,2025-12-14T21:15:24+00:00,https://www.facebook.com/permalink.php?story_f...,"𝑳𝒂 𝒅𝒊𝒗𝒆𝒓𝒔𝒊𝒕𝒆́ 𝒄𝒖𝒍𝒕𝒖𝒓𝒆𝒍𝒍𝒆. ✨\n\nÀ l’ENSEA, diff..."
3,AES_page,2025-10-14T12:40:52+00:00,https://www.facebook.com/permalink.php?story_f...,✨ « 𝐃𝐞 𝐧𝐨𝐬 𝐝𝐢𝐟𝐟𝐞́𝐫𝐞𝐧𝐜𝐞𝐬 𝐧𝐚𝐢̂𝐭𝐫𝐚 𝐧𝐨𝐭𝐫𝐞 𝐟𝐨𝐫𝐜𝐞. »...
4,AES_page,2025-10-05T14:28:48+00:00,https://www.facebook.com/permalink.php?story_f...,🎓 𝐂𝐞́𝐫𝐞́𝐦𝐨𝐧𝐢𝐞 𝐝𝐞 𝐝𝐢𝐩𝐥𝐨̂𝐦𝐚𝐭𝐢𝐨𝐧 – 𝐄𝐍𝐒𝐄𝐀 🎉\nCe 𝐬𝐚...


## Pourquoi un index séparé ?

Facebook contient des informations datées et moins institutionnelles. Les publications sont donc nettoyées, contrôlées et indexées séparément. Le routeur n'utilise cet index que lorsqu'une activité ou un club est explicitement nommé.


In [3]:
from getpass import getpass
from dataclasses import asdict
from facebook_scraper import FacebookScraper

RUN_FACEBOOK_COLLECTION = False
if RUN_FACEBOOK_COLLECTION:
    token = os.getenv('APIFY_TOKEN') or getpass('Jeton Apify : ')
    scraper = FacebookScraper(config, api_token=token)
    reports = scraper.run(max_posts_per_page=facebook.max_posts_per_page)
    display(pd.DataFrame([asdict(report) for report in reports]))
else:
    print('Collecte Apify désactivée ; les fichiers existants sont seulement inspectés.')


Collecte Apify désactivée ; les fichiers existants sont seulement inspectés.


In [4]:
from facebook_processor import FacebookPostProcessor

RUN_FACEBOOK_PREPARATION = True
if RUN_FACEBOOK_PREPARATION:
    report = FacebookPostProcessor(config).process_all()
    display(pd.Series(report).to_frame('valeur'))
else:
    print('Préparation désactivée : aucun chunk Facebook ne sera reconstruit.')


,valeur
raw_posts,232
posts_with_media,229
media_posts_without_usable_ocr,176
posts_with_usable_media_ocr,53
usable_media_ocr_fragments,84
removed_too_short,2
removed_duplicate_content,1
selected_posts,229
per_page,"{'AES_page': 28, 'CDR_page': 30, 'DAS_page': 2..."
cleaned_candidate_dir,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...


## Qualité minimale

Chaque publication conservée doit avoir un texte substantiel, une page d'origine, une date, un permalien et une empreinte unique. Les republications identiques et les textes trop courts sont retirés.
